# STORIES PFC dataset training

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Fit the three-age ordering model

In [ ]:
# QIAN PFC | STORIES author-tutorial run | GPU; no GPU memory preallocation.
# Three sections: GW15 UMB1117/F1b, GW20 FB080/F1, GW22 FB123/F2.
# Full 3-age fit for spatial trajectories; NOT the strict GW20 expression holdout.
# IMPORTANT: Run this single cell in a NEW/RESTARTED GPU-enabled Jupyter kernel.
# GPU config must precede ALL imports of jax, jaxlib, stories, or libraries that use JAX.

import os, sys
SMOKE = False             # True: 1,500 cells/section, 80 training steps.
STORIES_BACKEND = "cuda"  # Select NVIDIA CUDA explicitly; "gpu" may also try unavailable ROCm.

if "jax" in sys.modules:
    raise RuntimeError("JAX is already imported. RESTART the Jupyter kernel before running the GPU cell.")

os.environ["JAX_PLATFORMS"] = STORIES_BACKEND
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # JAX allocates GPU memory on demand.
os.environ.pop("XLA_PYTHON_CLIENT_MEM_FRACTION", None)
# Use all CPUs made available to this Python process by Slurm/CPU affinity.
# Do not artificially restrict numerical preprocessing to 2 threads.
try:
    AFFINITY_CPUS = len(os.sched_getaffinity(0))
except (AttributeError, OSError):
    AFFINITY_CPUS = os.cpu_count() or 1
SLURM_CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK") or AFFINITY_CPUS)
AVAILABLE_CPUS = max(1, min(AFFINITY_CPUS, SLURM_CPUS))
for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_var] = str(AVAILABLE_CPUS)

from pathlib import Path
import json, gc, time, random, hashlib, inspect, platform, subprocess
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
from sklearn.decomposition import PCA
import scanpy as sc
import jax, optax, stories
from flax.serialization import to_bytes

SEED = 19491001
random.seed(SEED)
np.random.seed(SEED)
PROJECT = Path(os.environ["STP_PROJECT"])
DATA_ROOT = Path(os.environ["QIAN_PFC_DATA_ROOT"])
# Dedicated GPU output: never overwrite/checkpoint-collide with the old CPU run.
OUT = PROJECT / "experiments/qian_pfc_publication_4way_20261024/stories_slices_1_2_7_tutorial_GPU"
if SMOKE:
    OUT = OUT.with_name(OUT.name + "_SMOKE")
assert DATA_ROOT.is_dir(), f"Missing Qian atlas: {DATA_ROOT}"
try:
    JAX_DEVICES = jax.devices()
except Exception as e:
    raise RuntimeError(
        "JAX GPU initialization failed. The kernel must be running in a Slurm GPU "
        "allocation with a CUDA-enabled JAX installation. Check nvidia-smi and "
        "jax.devices() in a fresh kernel."
    ) from e
if not JAX_DEVICES or not all(d.platform == "gpu" for d in JAX_DEVICES):
    raise RuntimeError(f"Expected GPU-only JAX backend; got {JAX_DEVICES}")

# Diagnostic only: host RAM is used on demand up to the Slurm allocation; no
# artificial per-process RAM cap is imposed by this runner.
def _cgroup_mem_limit_gib():
    for path in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            value = open(path).read().strip()
            if value.isdigit() and int(value) < (1 << 60):
                return round(int(value) / 2**30, 2)
        except OSError:
            pass
    return "unknown (check Slurm --mem allocation)"

print("STORIES backend:", STORIES_BACKEND, "devices:", JAX_DEVICES, flush=True)
print("JAX GPU VRAM preallocation: disabled (on-demand allocation)", flush=True)
print("CPU threads (Slurm/affinity):", AVAILABLE_CPUS,
      "| host RAM limit (GiB):", _cgroup_mem_limit_gib(), flush=True)
try:
    p = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10, check=False,
    )
    if p.returncode == 0:
        print("GPUs (total, used):", p.stdout.strip(), flush=True)
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

sys.path.insert(0, str(PROJECT / "src"))
from flow_stp.data.qian_pfc import (
    QianPFCConfig, build_qian_pfc_panel, read_h5ad_rows_direct,
)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h.update(b)
    return h.hexdigest()

def save_json(p, item):
    p.write_text(json.dumps(item, indent=2, sort_keys=True, default=str))

# Explicit nine-section inventory used to select the same 3 sections as stVCR.
cfg = QianPFCConfig(
    data_root=DATA_ROOT,
    targets=(("GW15", ("F1a", "F1b")),
             ("GW20", ("F1", "F2a", "F2b")),
             ("GW22", ("F1", "F2", "F3"))),
    expected_num_slices=9, layer_key_for_counts="raw",
)
panel = build_qian_pfc_panel(cfg)
triples = [(str(r.gw), str(r["sample"]), str(r.region))
           for _, r in panel.iterrows()]
expected_triplets = [
    ("GW15", "UMB1117", "F1a"),
    ("GW15", "UMB1117", "F1b"),
    ("GW20", "FB080", "F1"),
    ("GW20", "FB080", "F2a"),
    ("GW20", "FB080", "F2b"),
    ("GW20", "FB121", "F1"),
    ("GW22", "FB123", "F1"),
    ("GW22", "FB123", "F2"),
    ("GW22", "FB123", "F3"),
]
assert triples == expected_triplets, f"Nine-section panel changed: {triples}"
select = panel[panel.slice_id.isin([1, 2, 7])].copy().reset_index(drop=True)
assert select.slice_id.tolist() == [1, 2, 7]
print(select[["slice_id", "gw", "sample", "region", "n_full"]].to_string(index=False))
if not SMOKE:
    assert select.n_full.to_list() == [410492, 665204, 439986], "Full section counts changed"

if (OUT / "SUCCESS.json").is_file():
    print("Run already complete:", OUT / "SUCCESS.json", "(no retraining)")
else:
    # Do not discard any interrupted model/checkpoint after a restart.
    if OUT.exists() and any(OUT.iterdir()):
        raise RuntimeError(f"Output directory already contains files: {OUT}. Inspect or change OUT; will not overwrite.")
    OUT.mkdir(parents=True, exist_ok=True)
    panel.to_csv(OUT / "all_nine_inventory.csv", index=False)
    select.to_csv(OUT / "selected_three_tissues.csv", index=False)

    # Read only the three selected tissue panels. Read original raw MERFISH counts,
    # but compute author-style logCP10k expression PCs for STORIES.
    idx_by_section = []
    gene_names = None
    spatial_parts, count_parts, obs_parts = [], [], []
    n_all = sum((min(int(r.n_full), 1500) if SMOKE else int(r.n_full)) for _, r in select.iterrows())
    X_log = np.empty((n_all, 300), dtype=np.float32)
    offset = 0
    for _, r in select.iterrows():
        path = Path(r.path)
        backed = sc.read_h5ad(path, backed="r")
        try:
            assert backed.raw is not None, f"Missing raw count panel: {path}"
            genes = backed.raw.var_names.astype(str).tolist()
            assert len(genes) == 300, (r.gw, len(genes))
            if gene_names is None:
                gene_names = genes
            else:
                assert genes == gene_names, "Gene ordering differs across stages"
            mask = ((backed.obs["region"].astype(str).to_numpy() == str(r.region)) &
                    (backed.obs["sample"].astype(str).to_numpy() == str(r["sample"])))
            idx = np.flatnonzero(mask)
            assert len(idx) == int(r.n_full), (r.slice_id, len(idx), r.n_full)
            if SMOKE:
                rng = np.random.default_rng(SEED + int(r.slice_id))
                idx = np.sort(rng.choice(idx, size=min(1500, len(idx)), replace=False))
            xy = np.asarray(backed.obsm["spatial"], dtype=np.float32)[idx, :2]
            assert np.isfinite(xy).all(), f"Bad coordinates in slice {r.slice_id}"
            cell_ids = np.asarray(backed.obs_names.astype(str))[idx]
            extras = {}
            for field in ("H1_annotation", "H2_annotation", "H3_annotation"):
                if field in backed.obs:
                    extras[field] = backed.obs.iloc[idx][field].astype(str).to_numpy(dtype=object)
        finally:
            backed.file.close()

        raw = read_h5ad_rows_direct(path, idx, layer_key="raw")
        assert raw.shape == (len(idx), 300) and np.isfinite(raw).all()
        assert np.min(raw) >= 0 and np.allclose(raw[:min(10000,len(raw))], np.rint(raw[:min(10000,len(raw))])), "Expected raw integer count data"
        count_parts.append(sp.csr_matrix(raw))
        row_sums = raw.sum(axis=1)
        for j in range(0, len(idx), 8192):
            b = raw[j:j+8192]
            scale = (10000.0 / np.maximum(row_sums[j:j+8192], 1.0)).astype(np.float32)
            b *= scale[:, None]
            np.log1p(b, out=b)
        X_log[offset:offset+len(idx)] = raw
        offset += len(idx)
        spatial_parts.append(xy)
        obs_parts.append(pd.DataFrame({
            "gw": str(r.gw), "sample": str(r["sample"]), "region": str(r.region),
            "original_slice_id": int(r.slice_id), "time": (float(r.time) - 15.0) / 7.0,
            "cell_id": cell_ids.astype(object),
            **extras,
        }))
        del raw
        gc.collect()
        print(f"Loaded {r.gw} {r['sample']}/{r.region}: {len(idx):,} cells", flush=True)

    assert offset == n_all and np.isfinite(X_log).all()
    assert len(gene_names) == 300
    print("Fitting public-only 20-dimensional PCA on logCP10k...", flush=True)
    pca = PCA(n_components=20, svd_solver="randomized", random_state=SEED, iterated_power=3)
    scores = pca.fit_transform(X_log).astype(np.float32)
    del X_log
    gc.collect()
    pc_scale = float(scores.max())  # Authors' tutorial: signed/global maximum.
    assert pc_scale > 0 and np.isfinite(pc_scale)
    scores /= pc_scale

    obs = pd.concat(obs_parts, ignore_index=True)
    obs.index = pd.Index(np.asarray([f"s{s}:{c}" for s,c in zip(obs.original_slice_id, obs.cell_id)], dtype=object), dtype=object, name="cell_key")
    assert obs.index.is_unique
    var = pd.DataFrame(index=pd.Index(np.asarray(gene_names, dtype=object), dtype=object, name="gene"))
    data = ad.AnnData(X=sp.vstack(count_parts, format="csr"), obs=obs, var=var)
    data.obsm["X_pca_harmony"] = scores  # Key from author tutorial; ordinary PCA here, NOT Harmony.
    data.obsm["spatial_raw"] = np.vstack(spatial_parts).astype(np.float32)
    space = data.obsm["spatial_raw"].copy()
    spatial_means = np.zeros((3, 2)); spatial_stds = np.zeros((3, 2))
    for k, sid in enumerate([1, 2, 7]):
        q = data.obs.original_slice_id.to_numpy() == sid
        spatial_means[k] = space[q].mean(axis=0, dtype=np.float64)
        spatial_stds[k] = space[q].std(axis=0, dtype=np.float64)
        assert np.all(spatial_stds[k] > 0), f"Degenerate tissue {sid}"
        space[q] = (space[q] - spatial_means[k]) / spatial_stds[k]
    data.obsm["spatial"] = space
    assert np.all(np.isfinite(space)) and len(np.unique(data.obs.time)) == 3
    assert np.allclose(np.sort(data.obs.time.unique()), [0., 5./7., 1.])
    # AnnData/HDF5 compatibility with pandas StringDtype and named indices.
    for df in (data.obs, data.var):
        df.index = pd.Index(np.asarray(df.index.tolist(), dtype=object), dtype=object, name=df.index.name)
        for col in df.columns:
            if isinstance(df[col].dtype, pd.StringDtype):
                df[col] = df[col].astype(object)

    np.savez_compressed(OUT / "preprocessing_frozen.npz",
        genes=np.asarray(gene_names), pca_mean=pca.mean_,
        pca_components=pca.components_, pca_scale=pc_scale,
        spatial_means=spatial_means, spatial_stds=spatial_stds,
        section_ids=np.array([1, 2, 7]), times=np.array([0., 5./7., 1.]))
    data.obs[["gw","sample","region","original_slice_id","time","cell_id"]].to_csv(OUT / "selected_cells.csv.gz")
    print("Saving STORIES input AnnData...", flush=True)
    data.write_h5ad(OUT / "stories_input_3tissue.h5ad", compression="gzip")

    config = dict(
        method="STORIES", benchmark="Qian PFC full three-tissue trajectory run",
        training_sections=[1,2,7], dataset_cells=int(data.n_obs), genes=300,
        model_class="stories.SpaceTime(quadratic_weight=1e-3)",
        model_weight_type="author vignette, not global API default",
        time_scale="(gestational_week-15)/7",
        expression="300 raw counts -> library-normalized logCP10k -> sklearn randomized PCA20 (no Harmony) -> divide by max PCA score",
        spatial="per-physical-section center and axiswise standard deviation, observed geometry only",
        weights="uniform (no growth prior)",
        opt="AdamW cosine_decay_schedule(1e-2,10000)",
        fit={"max_iter":80 if SMOKE else 10000,
             "batch_size":128 if SMOKE else 1000,
             "train_val_split":0.75,"patience":20 if SMOKE else 150,
             "min_delta":0.0,"restore":True},
        smoke=SMOKE, backend=STORIES_BACKEND,
        gpu_mem_fraction=None,
        gpu_preallocate=False, cpu_threads=AVAILABLE_CPUS,
        slurm_mem_gib=_cgroup_mem_limit_gib(), seed=SEED,
        python=sys.version, jax=jax.__version__, stories_module=inspect.getsourcefile(stories.SpaceTime),
        output=str(OUT), dataset_root=str(DATA_ROOT), host=platform.node())
    save_json(OUT / "run_config.json", config)
    print("Starting official STORIES tutorial fit...", flush=True)
    model = stories.SpaceTime(quadratic_weight=1e-3)
    start = time.perf_counter()
    model.fit(
        adata=data, time_key="time", omics_key="X_pca_harmony", space_key="spatial",
        optimizer=optax.adamw(optax.cosine_decay_schedule(1e-2, 10000)),
        max_iter=80 if SMOKE else 10000,
        batch_size=128 if SMOKE else 1000,
        train_val_split=0.75, patience=20 if SMOKE else 150,
        min_delta=0.0, restore=True,
        checkpoint_manager=str((OUT / "ckpt_stories").resolve()),
    )
    seconds = time.perf_counter() - start
    with open(OUT / "native_best_params.msgpack", "wb") as f:
        f.write(to_bytes(model.params))
    def host_float_array(xs):
        return np.asarray([float(np.asarray(jax.device_get(v))) for v in xs], dtype=np.float64)
    np.savez_compressed(OUT / "training_history.npz",
        train_it=np.asarray(model.train_it),
        train_loss=host_float_array(model.train_losses),
        val_it=np.asarray(model.val_it),
        val_loss=host_float_array(model.val_losses),
        best_step=np.asarray(model.best_step))

    # Native QC on 2k observed cells per section; a diagnostic failure must not
    # erase an otherwise successful expensive training run.
    qc_status = "success"
    try:
        qc_rng = np.random.default_rng(SEED)
        qc_ids = np.concatenate([qc_rng.choice(np.flatnonzero(data.obs.original_slice_id.to_numpy() == sid),
                              size=min(2000, int((data.obs.original_slice_id == sid).sum())), replace=False)
                              for sid in [1,2,7]])
        qc = data[qc_ids].copy()
        stories.tools.compute_potential(qc, model, "X_pca_harmony")
        np.savez_compressed(OUT / "native_potential_QC.npz",
            cell_key=np.asarray(qc.obs_names, dtype=str),
            time=qc.obs.time.to_numpy(float),
            potential=qc.obs.potential.to_numpy(float),
            spatial=qc.obsm["spatial_raw"].astype(np.float32))
    except Exception as e:
        qc_status = "failed: " + repr(e)
        print("QC potential export failed; trained model/checkpoint are intact:", repr(e), flush=True)

    save_json(OUT / "SUCCESS.json", {
        "status":"success", "smoke":SMOKE, "seconds":seconds,
        "n_cells":int(data.n_obs), "n_genes":int(data.n_vars),
        "best_step":int(model.best_step), "potential_qc":qc_status,
        "model_sha256":sha256(OUT / "native_best_params.msgpack"),
        "preprocessing_sha256":sha256(OUT / "preprocessing_frozen.npz"),
        "input_sha256":sha256(OUT / "stories_input_3tissue.h5ad"),
        "note":"All three GW ages present in training; NOT a strict GW20-heldout forecast.",
    })
    print("STORIES COMPLETED:", OUT, "best_step=", model.best_step, flush=True)

## Fit the strict GW20 holdout model

In [ ]:
# QIAN PFC | STRICT GW20 AGE HOLDOUT | STORIES author-tutorial GPU trainer.
# Training data: ONLY GW15 UMB1117/F1b and GW22 FB123/F2 (850,478 cells).
# ALL GW20 samples remain completely unread until a separate evaluation run.
# Distinct run from full-data three-age model; full author settings retained.
# IMPORTANT: Run this single cell in a NEW/RESTARTED GPU-enabled Jupyter kernel.
# GPU config must precede ALL imports of jax, jaxlib, stories, or libraries that use JAX.

import os, sys
SMOKE = False             # True: 1,500 cells/section, 80 training steps.
STORIES_BACKEND = "cuda"  # Select NVIDIA CUDA explicitly; "gpu" may also try unavailable ROCm.

if "jax" in sys.modules:
    raise RuntimeError("JAX is already imported. RESTART the Jupyter kernel before running the GPU cell.")

os.environ["JAX_PLATFORMS"] = STORIES_BACKEND
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # JAX allocates GPU memory on demand.
os.environ.pop("XLA_PYTHON_CLIENT_MEM_FRACTION", None)
# Use all CPUs made available to this Python process by Slurm/CPU affinity.
# Do not artificially restrict numerical preprocessing to 2 threads.
try:
    AFFINITY_CPUS = len(os.sched_getaffinity(0))
except (AttributeError, OSError):
    AFFINITY_CPUS = os.cpu_count() or 1
SLURM_CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK") or AFFINITY_CPUS)
AVAILABLE_CPUS = max(1, min(AFFINITY_CPUS, SLURM_CPUS))
for _thread_var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_thread_var] = str(AVAILABLE_CPUS)

from pathlib import Path
import json, gc, time, random, hashlib, inspect, platform, subprocess
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
from sklearn.decomposition import PCA
import scanpy as sc
import jax, optax, stories
from flax.serialization import to_bytes

SEED = 19491001
random.seed(SEED)
np.random.seed(SEED)
PROJECT = Path(os.environ["STP_PROJECT"])
DATA_ROOT = Path(os.environ["QIAN_PFC_DATA_ROOT"])
# New held-out GPU output: cannot conflict with full three-stage models.
OUT = PROJECT / "experiments/qian_pfc_publication_4way_20261024/stories_gw15_gw22_STRICT_GW20_HELDOUT_tutorial_GPU"
if SMOKE:
    OUT = OUT.with_name(OUT.name + "_SMOKE")
assert DATA_ROOT.is_dir(), f"Missing Qian atlas: {DATA_ROOT}"
try:
    JAX_DEVICES = jax.devices()
except Exception as e:
    raise RuntimeError(
        "JAX GPU initialization failed. The kernel must be running in a Slurm GPU "
        "allocation with a CUDA-enabled JAX installation. Check nvidia-smi and "
        "jax.devices() in a fresh kernel."
    ) from e
if not JAX_DEVICES or not all(d.platform == "gpu" for d in JAX_DEVICES):
    raise RuntimeError(f"Expected GPU-only JAX backend; got {JAX_DEVICES}")

# Diagnostic only: host RAM is used on demand up to the Slurm allocation; no
# artificial per-process RAM cap is imposed by this runner.
def _cgroup_mem_limit_gib():
    for path in ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            value = open(path).read().strip()
            if value.isdigit() and int(value) < (1 << 60):
                return round(int(value) / 2**30, 2)
        except OSError:
            pass
    return "unknown (check Slurm --mem allocation)"

print("STORIES backend:", STORIES_BACKEND, "devices:", JAX_DEVICES, flush=True)
print("JAX GPU VRAM preallocation: disabled (on-demand allocation)", flush=True)
print("CPU threads (Slurm/affinity):", AVAILABLE_CPUS,
      "| host RAM limit (GiB):", _cgroup_mem_limit_gib(), flush=True)
try:
    p = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=10, check=False,
    )
    if p.returncode == 0:
        print("GPUs (total, used):", p.stdout.strip(), flush=True)
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

sys.path.insert(0, str(PROJECT / "src"))
from flow_stp.data.qian_pfc import (
    QianPFCConfig, build_qian_pfc_panel, read_h5ad_rows_direct,
)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h.update(b)
    return h.hexdigest()

def save_json(p, item):
    p.write_text(json.dumps(item, indent=2, sort_keys=True, default=str))

# This is a STRICT age holdout. The panel builder opens ONLY gw15.h5ad
# and gw22.h5ad, NEVER gw20.h5ad, not even for its metadata/inventory.
# Local two-section panel IDs are remapped to original benchmark IDs [1,7].
cfg = QianPFCConfig(
    data_root=DATA_ROOT,
    targets=(("GW15", ("F1b",)), ("GW22", ("F2",))),
    expected_num_slices=2, layer_key_for_counts="raw",
)
panel = build_qian_pfc_panel(cfg)
triples = [(str(r.gw), str(r["sample"]), str(r.region))
           for _, r in panel.iterrows()]
EXPECTED = [("GW15", "UMB1117", "F1b", 410492, 1),
            ("GW22", "FB123", "F2", 439986, 7)]
assert triples == [(a,b,c) for a,b,c,_,_ in EXPECTED], (
    f"Training-only panel changed or included unintended tissues: {triples}")
select = panel.copy().reset_index(drop=True)
select["local_panel_slice_id"] = select["slice_id"]
select["slice_id"] = [1,7]  # Stable original nine-section benchmark identifiers.
assert select.slice_id.tolist() == [1,7]
assert [int(n) for n in select.n_full] == [410492,439986]
assert all(Path(p).name in {"gw15.h5ad", "gw22.h5ad"} for p in select.path)
assert all("GW20" not in str(item).upper() for item in select.gw)
print("STRICT HOLDOUT TRAINING ONLY: NO GW20 input files will be opened")
print(select[["slice_id", "gw", "sample", "region", "n_full"]].to_string(index=False),flush=True)
if not SMOKE:
    assert sum(int(n) for n in select.n_full) == 850478

if (OUT / "SUCCESS.json").is_file():
    print("Run already complete:", OUT / "SUCCESS.json", "(no retraining)")
else:
    # Do not discard any interrupted model/checkpoint after a restart.
    if OUT.exists() and any(OUT.iterdir()):
        raise RuntimeError(f"Output directory already contains files: {OUT}. Inspect or change OUT; will not overwrite.")
    OUT.mkdir(parents=True, exist_ok=True)
    # Inventory contains only training tissues; no age-20 data or metadata.
    panel.to_csv(OUT / "training_only_two_section_inventory.csv", index=False)
    select.to_csv(OUT / "selected_two_tissues.csv", index=False)

    # Read only the two allowed training tissue panels. Read original raw MERFISH counts,
    # but compute author-style logCP10k expression PCs for STORIES.
    idx_by_section = []
    gene_names = None
    spatial_parts, count_parts, obs_parts = [], [], []
    n_all = sum((min(int(r.n_full), 1500) if SMOKE else int(r.n_full)) for _, r in select.iterrows())
    X_log = np.empty((n_all, 300), dtype=np.float32)
    offset = 0
    for _, r in select.iterrows():
        path = Path(r.path)
        backed = sc.read_h5ad(path, backed="r")
        try:
            assert backed.raw is not None, f"Missing raw count panel: {path}"
            genes = backed.raw.var_names.astype(str).tolist()
            assert len(genes) == 300, (r.gw, len(genes))
            if gene_names is None:
                gene_names = genes
            else:
                assert genes == gene_names, "Gene ordering differs across stages"
            mask = ((backed.obs["region"].astype(str).to_numpy() == str(r.region)) &
                    (backed.obs["sample"].astype(str).to_numpy() == str(r["sample"])))
            idx = np.flatnonzero(mask)
            assert len(idx) == int(r.n_full), (r.slice_id, len(idx), r.n_full)
            if SMOKE:
                rng = np.random.default_rng(SEED + int(r.slice_id))
                idx = np.sort(rng.choice(idx, size=min(1500, len(idx)), replace=False))
            xy = np.asarray(backed.obsm["spatial"], dtype=np.float32)[idx, :2]
            assert np.isfinite(xy).all(), f"Bad coordinates in slice {r.slice_id}"
            cell_ids = np.asarray(backed.obs_names.astype(str))[idx]
            extras = {}
            for field in ("H1_annotation", "H2_annotation", "H3_annotation"):
                if field in backed.obs:
                    extras[field] = backed.obs.iloc[idx][field].astype(str).to_numpy(dtype=object)
        finally:
            backed.file.close()

        raw = read_h5ad_rows_direct(path, idx, layer_key="raw")
        assert raw.shape == (len(idx), 300) and np.isfinite(raw).all()
        assert np.min(raw) >= 0 and np.allclose(raw[:min(10000,len(raw))], np.rint(raw[:min(10000,len(raw))])), "Expected raw integer count data"
        count_parts.append(sp.csr_matrix(raw))
        row_sums = raw.sum(axis=1)
        for j in range(0, len(idx), 8192):
            b = raw[j:j+8192]
            scale = (10000.0 / np.maximum(row_sums[j:j+8192], 1.0)).astype(np.float32)
            b *= scale[:, None]
            np.log1p(b, out=b)
        X_log[offset:offset+len(idx)] = raw
        offset += len(idx)
        spatial_parts.append(xy)
        obs_parts.append(pd.DataFrame({
            "gw": str(r.gw), "sample": str(r["sample"]), "region": str(r.region),
            "original_slice_id": int(r.slice_id), "time": (float(r.time) - 15.0) / 7.0,
            "cell_id": cell_ids.astype(object),
            **extras,
        }))
        del raw
        gc.collect()
        print(f"Loaded {r.gw} {r['sample']}/{r.region}: {len(idx):,} cells", flush=True)

    assert offset == n_all and np.isfinite(X_log).all()
    assert len(gene_names) == 300
    print("Fitting public-only 20-dimensional PCA on logCP10k...", flush=True)
    pca = PCA(n_components=20, svd_solver="randomized", random_state=SEED, iterated_power=3)
    scores = pca.fit_transform(X_log).astype(np.float32)
    del X_log
    gc.collect()
    pc_scale = float(scores.max())  # Authors' tutorial: signed/global maximum.
    assert pc_scale > 0 and np.isfinite(pc_scale)
    scores /= pc_scale

    obs = pd.concat(obs_parts, ignore_index=True)
    obs.index = pd.Index(np.asarray([f"s{s}:{c}" for s,c in zip(obs.original_slice_id, obs.cell_id)], dtype=object), dtype=object, name="cell_key")
    assert obs.index.is_unique
    var = pd.DataFrame(index=pd.Index(np.asarray(gene_names, dtype=object), dtype=object, name="gene"))
    data = ad.AnnData(X=sp.vstack(count_parts, format="csr"), obs=obs, var=var)
    data.obsm["X_pca_harmony"] = scores  # Key from author tutorial; ordinary PCA here, NOT Harmony.
    data.obsm["spatial_raw"] = np.vstack(spatial_parts).astype(np.float32)
    space = data.obsm["spatial_raw"].copy()
    spatial_means = np.zeros((2, 2)); spatial_stds = np.zeros((2, 2))
    for k, sid in enumerate([1, 7]):
        q = data.obs.original_slice_id.to_numpy() == sid
        spatial_means[k] = space[q].mean(axis=0, dtype=np.float64)
        spatial_stds[k] = space[q].std(axis=0, dtype=np.float64)
        assert np.all(spatial_stds[k] > 0), f"Degenerate tissue {sid}"
        space[q] = (space[q] - spatial_means[k]) / spatial_stds[k]
    data.obsm["spatial"] = space
    assert np.all(np.isfinite(space)) and len(np.unique(data.obs.time)) == 2
    assert np.allclose(np.sort(data.obs.time.unique()), [0., 1.])
    # AnnData/HDF5 compatibility with pandas StringDtype and named indices.
    for df in (data.obs, data.var):
        df.index = pd.Index(np.asarray(df.index.tolist(), dtype=object), dtype=object, name=df.index.name)
        for col in df.columns:
            if isinstance(df[col].dtype, pd.StringDtype):
                df[col] = df[col].astype(object)

    np.savez_compressed(OUT / "preprocessing_frozen.npz",
        genes=np.asarray(gene_names), pca_mean=pca.mean_,
        pca_components=pca.components_, pca_scale=pc_scale,
        spatial_means=spatial_means, spatial_stds=spatial_stds,
        section_ids=np.array([1, 7]), times=np.array([0., 1.]))
    data.obs[["gw","sample","region","original_slice_id","time","cell_id"]].to_csv(OUT / "selected_cells.csv.gz")
    print("Saving STORIES input AnnData...", flush=True)
    data.write_h5ad(OUT / "stories_input_GW15_GW22_ONLY.h5ad", compression="gzip")

    config = dict(
        method="STORIES", benchmark="Qian PFC STRICT held-out-GW20 expression run",
        training_sections=[1,7], dataset_cells=int(data.n_obs), genes=300,
        model_class="stories.SpaceTime(quadratic_weight=1e-3)",
        model_weight_type="author vignette, not global API default",
        time_scale="(gestational_week-15)/7",
        completely_heldout_ages=["GW20"],
        heldout_evaluation_time=5.0/7.0,
        train_only_pca=True,
        train_only_spatial_scaling=True,
        source_H5AD_only=["gw15.h5ad", "gw22.h5ad"],
        expression="300 raw counts -> library-normalized logCP10k -> sklearn randomized PCA20 (no Harmony) -> divide by max PCA score",
        spatial="per-physical-section center and axiswise standard deviation, observed geometry only",
        weights="uniform (no growth prior)",
        opt="AdamW cosine_decay_schedule(1e-2,10000)",
        fit={"max_iter":80 if SMOKE else 10000,
             "batch_size":128 if SMOKE else 1000,
             "train_val_split":0.75,"patience":20 if SMOKE else 150,
             "min_delta":0.0,"restore":True},
        smoke=SMOKE, backend=STORIES_BACKEND,
        gpu_mem_fraction=None,
        gpu_preallocate=False, cpu_threads=AVAILABLE_CPUS,
        slurm_mem_gib=_cgroup_mem_limit_gib(), seed=SEED,
        python=sys.version, jax=jax.__version__, stories_module=inspect.getsourcefile(stories.SpaceTime),
        output=str(OUT), dataset_root=str(DATA_ROOT), host=platform.node())
    save_json(OUT / "run_config.json", config)
    print("Starting official STORIES tutorial fit...", flush=True)
    model = stories.SpaceTime(quadratic_weight=1e-3)
    start = time.perf_counter()
    model.fit(
        adata=data, time_key="time", omics_key="X_pca_harmony", space_key="spatial",
        optimizer=optax.adamw(optax.cosine_decay_schedule(1e-2, 10000)),
        max_iter=80 if SMOKE else 10000,
        batch_size=128 if SMOKE else 1000,
        train_val_split=0.75, patience=20 if SMOKE else 150,
        min_delta=0.0, restore=True,
        checkpoint_manager=str((OUT / "ckpt_stories").resolve()),
    )
    seconds = time.perf_counter() - start
    with open(OUT / "native_best_params.msgpack", "wb") as f:
        f.write(to_bytes(model.params))
    def host_float_array(xs):
        return np.asarray([float(np.asarray(jax.device_get(v))) for v in xs], dtype=np.float64)
    np.savez_compressed(OUT / "training_history.npz",
        train_it=np.asarray(model.train_it),
        train_loss=host_float_array(model.train_losses),
        val_it=np.asarray(model.val_it),
        val_loss=host_float_array(model.val_losses),
        best_step=np.asarray(model.best_step))

    # Native QC on 2k observed cells per section; a diagnostic failure must not
    # erase an otherwise successful expensive training run.
    qc_status = "success"
    try:
        qc_rng = np.random.default_rng(SEED)
        qc_ids = np.concatenate([qc_rng.choice(np.flatnonzero(data.obs.original_slice_id.to_numpy() == sid),
                              size=min(2000, int((data.obs.original_slice_id == sid).sum())), replace=False)
                              for sid in [1,7]])
        qc = data[qc_ids].copy()
        stories.tools.compute_potential(qc, model, "X_pca_harmony")
        np.savez_compressed(OUT / "native_potential_QC.npz",
            cell_key=np.asarray(qc.obs_names, dtype=str),
            time=qc.obs.time.to_numpy(float),
            potential=qc.obs.potential.to_numpy(float),
            spatial=qc.obsm["spatial_raw"].astype(np.float32))
    except Exception as e:
        qc_status = "failed: " + repr(e)
        print("QC potential export failed; trained model/checkpoint are intact:", repr(e), flush=True)

    save_json(OUT / "SUCCESS.json", {
        "status":"success", "smoke":SMOKE, "seconds":seconds,
        "n_cells":int(data.n_obs), "n_genes":int(data.n_vars),
        "training_sections":[1,7],"heldout_age":"GW20",
        "best_step":int(model.best_step), "potential_qc":qc_status,
        "model_sha256":sha256(OUT / "native_best_params.msgpack"),
        "preprocessing_sha256":sha256(OUT / "preprocessing_frozen.npz"),
        "input_sha256":sha256(OUT / "stories_input_GW15_GW22_ONLY.h5ad"),
        "note":"STRICT GW20 HOLDOUT: training inputs only GW15 + GW22, never GW20. Model is ready for future independent held-out evaluation.",
        "never_loaded_GW20":True, "n_train_ages":2,
    })
    print("STORIES COMPLETED:", OUT, "best_step=", model.best_step, flush=True)

## Export the strict holdout forecast

In [ ]:
# Qian PFC | STORIES native held-out GW20 expression prediction
# Paste this ONE CELL into the STORIES PFC training notebook, under its native kernel.
# NO TRAINING. NO GW20 DATA READ. Uses the newest STRICT held-out checkpoint.

import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

from pathlib import Path
import hashlib
import json
import gc
import numpy as np
import scanpy as sc
import anndata as ad
import jax
import jax.numpy as jnp
import stories
from flax import serialization

PROJECT = Path(os.environ["STP_PROJECT"])
EXPERIMENTS = PROJECT / 'experiments/qian_pfc_publication_4way_20261024'
TRAIN = EXPERIMENTS / 'stories_gw15_gw22_STRICT_GW20_HELDOUT_tutorial_GPU'
PRED_DIR = (EXPERIMENTS /
    'GW20_heldout_fourway_Fisher_flow_vs_native_v1/frozen_predictions')
PRED_FILE = PRED_DIR / 'stories.npz'
TI = (PROJECT /
    'experiments/qian_pfc_gw20_allheldout_gw15_gw22_five_tissue_v1/CPU_FULL')

SEED = 20261027  # Shared comparison seed
N_PRED = 4096
GW20_TIME = 5.0 / 7.0  # Training clock: (GW - 15) / 7


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def required(path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing required frozen artifact: {path}')
    return path


# ------------------------------------------------------------
# 1. Validate the NEWEST checkpoint and strict age holdout
# ------------------------------------------------------------
if PRED_FILE.exists():
    raise FileExistsError(f'STORIES prediction already frozen; refusing overwrite: {PRED_FILE}')

success = json.loads(required(TRAIN / 'SUCCESS.json').read_text())
assert success.get('status') == 'success', success
assert success.get('smoke') is False, 'Refusing smoke-test checkpoint'
assert success.get('training_sections') == [1, 7], 'Wrong training tissues'
assert int(success.get('n_cells', -1)) == 850478, 'Wrong training cell count'
assert int(success.get('n_genes', -1)) == 300, 'Wrong gene count'
assert success.get('heldout_age') == 'GW20'
assert success.get('never_loaded_GW20') is True

CKPT = required(TRAIN / 'native_best_params.msgpack')
PREP = required(TRAIN / 'preprocessing_frozen.npz')
INPUT = required(TRAIN / 'stories_input_GW15_GW22_ONLY.h5ad')
canonical_genes = json.loads(required(TI / 'gene_names.json').read_text())
assert len(canonical_genes) == len(set(canonical_genes)) == 300

assert sha256(CKPT) == success['model_sha256'], 'STORIES checkpoint hash mismatch'
assert sha256(PREP) == success['preprocessing_sha256'], 'PCA artifact hash mismatch'

with np.load(PREP, allow_pickle=False) as f:
    genes = f['genes'].astype(str).tolist()
    pca_components = np.asarray(f['pca_components'], dtype=np.float64)
    pca_mean = np.asarray(f['pca_mean'], dtype=np.float64)
    pca_scale = float(f['pca_scale'])
    training_section_ids = f['section_ids'].astype(int).tolist()
    training_times = np.asarray(f['times'], dtype=float)

assert genes == canonical_genes, '300-gene ordering differs from Spatial TI'
assert pca_components.shape == (20, 300)
assert pca_mean.shape == (300,)
assert np.isfinite(pca_scale) and pca_scale > 0
assert training_section_ids == [1, 7]
assert np.allclose(training_times, [0.0, 1.0])

print('STORIES native GW20 prediction — newest strict age holdout')
print('Source: GW15 Tissue 2; training also used GW22 Tissue 2')
print('JAX devices:', jax.devices())
print('Target normalized time:', GW20_TIME)

# ------------------------------------------------------------
# 2. Read ONLY the training-only PCA coordinates, not counts
# ------------------------------------------------------------
adata = sc.read_h5ad(INPUT, backed='r')
try:
    assert adata.n_obs == 850478 and adata.n_vars == 300
    assert adata.var_names.astype(str).tolist() == genes
    section = adata.obs['original_slice_id'].to_numpy(dtype=int)
    assert set(np.unique(section)) == {1, 7}
    assert int(np.sum(section == 1)) == 410492
    assert int(np.sum(section == 7)) == 439986

    source_rows = np.flatnonzero(section == 1)
    rng = np.random.default_rng(SEED + 222)
    chosen = np.sort(rng.choice(source_rows, size=N_PRED, replace=False))
    x_initial = np.asarray(
        adata.obsm['X_pca_harmony'][chosen], dtype=np.float32
    ).copy()
    assert x_initial.shape == (4096, 20)
    assert np.isfinite(x_initial).all()
finally:
    adata.file.close()
    del adata
    gc.collect()

print('Starting GW15 cells:', len(x_initial))

# ------------------------------------------------------------
# 3. Restore saved native STORIES potential, NO model.fit()
# ------------------------------------------------------------
model = stories.SpaceTime(quadratic_weight=1e-3)
# Flax requires a matching parameter-structure template for deserialization.
# It is replaced completely by the trained parameter bytes below.
template = model.potential.init(
    jax.random.PRNGKey(0), jnp.ones((1000, 20), dtype=jnp.float32)
)
model.params = serialization.from_bytes(template, CKPT.read_bytes())

# ------------------------------------------------------------
# 4. Native expression-state transport GW15 -> GW20
# ------------------------------------------------------------
start_adata = ad.AnnData(X=np.zeros((N_PRED, 1), dtype=np.float32))
start_adata.obsm['X_pca_harmony'] = x_initial

print('Running native SpaceTime.transform(..., tau=5/7)...', flush=True)
y_latent = np.asarray(
    model.transform(
        start_adata,
        omics_key='X_pca_harmony',
        tau=GW20_TIME,
        batch_size=512,
        key=jax.random.PRNGKey(SEED),
    ), dtype=np.float64
)

assert y_latent.shape == (N_PRED, 20), y_latent.shape
assert np.isfinite(y_latent).all(), 'Nonfinite STORIES forecast'

# ------------------------------------------------------------
# 5. Decode with the TRAINING-ONLY PCA; convert to gene shares
# ------------------------------------------------------------
# Training representation: logCP10k -> PCA20 -> / global max score.
# Inverse PCA is an explicit *evaluation adapter*, not a native count decoder.
logcp10k_pred = (
    (y_latent * pca_scale) @ pca_components + pca_mean[None, :]
)
expression = np.expm1(np.clip(logcp10k_pred, -30, 30))
negative_fraction = float(np.mean(expression < 0))
expression = np.maximum(expression, 0)

assert np.isfinite(expression).all()
assert np.all(expression.sum(axis=1) > 0), 'Zero-mass gene vector after PCA inversion'
shares = (expression / expression.sum(axis=1, keepdims=True)).astype(np.float32)
assert shares.shape == (N_PRED, 300)
assert np.allclose(shares.sum(axis=1), 1.0, atol=1e-5)

metadata = dict(
    method='STORIES',
    source_age='GW15', target_age='GW20',
    source_t=0.0, target_t_normalized=GW20_TIME,
    training_sections=2, train_n_cells=850478,
    training_tissue_labels=['GW15 Tissue 2', 'GW22 Tissue 2'],
    source_tissue_label='GW15 Tissue 2',
    prediction_rule='author SpaceTime.transform on native PCA20 states',
    decoder='training-only inverse PCA20, inverse log1p, clip negatives',
    negative_decoded_fraction=negative_fraction,
    checkpoint=str(CKPT), checkpoint_sha256=sha256(CKPT),
    preprocessing_sha256=sha256(PREP), seed=SEED,
    n_pred=N_PRED, n_genes=300, gw20_observations_read=False,
)

# ------------------------------------------------------------
# 6. Freeze common-format forecast, never overwrite existing
# ------------------------------------------------------------
PRED_DIR.mkdir(parents=True, exist_ok=True)
tmp = PRED_DIR / 'stories.npz.incomplete'
if tmp.exists():
    raise FileExistsError(f'Leftover incomplete output; inspect before rerunning: {tmp}')
try:
    with tmp.open('xb') as f:
        np.savez_compressed(
            f,
            shares=shares,
            genes=np.asarray(genes),
            meta_json=np.array(json.dumps(metadata, sort_keys=True)),
            source_training_rows=chosen.astype(np.int64),
        )
    with np.load(tmp, allow_pickle=False) as f:
        assert f['shares'].shape == (4096, 300)
        assert f['genes'].astype(str).tolist() == canonical_genes
    # Hard-link creates the final frozen name atomically without overwriting.
    os.link(tmp, PRED_FILE)
finally:
    if tmp.exists():
        tmp.unlink()

print('\n' + '=' * 66)
print('STORIES GW20 PREDICTION FROZEN:', PRED_FILE)
print('SHA-256:', sha256(PRED_FILE))
print('Shape:', shares.shape)
print('Negative pre-clipping fraction:', negative_fraction)
print('GW20 observations accessed: NONE')
print('Next: moscot native prediction, then common held-out scoring.')